In [ ]:
from ipaddress import v4_int_to_packed

først utkast Kø
import streamlit as st
from datetime import datetime, timedelta

# 1. Konfigurasjon av siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

st.title("🖨️ Bambulab Tillitsbasert Køsystem")
st.write("Printeren går fra 05:00 til 01:00 (20 timer / 20 tokens). 1 token ≈ 1 time.")

# 2. Sette opp "State" (Streamlit husker data mellom trykk)
if "tokens" not in st.session_state:
    # Lager 20 ledige slots fra kl 05:00 og utover
    start_tid = datetime.strptime("05:00", "%H:%M")
    st.session_state.tokens = []
    for i in range(20):
        tidspunkt = (start_tid + timedelta(hours=i)).strftime("%H:%M")
        st.session_state.tokens.append({"id": i + 1, "tid": tidspunkt, "bruker": "Ledig", "status": "Ledig"})

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

# 3. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Ole")
antall_tokens = st.number_input("Hvor mange tokens trenger du? (1 token = 1 time)", min_value=1, max_value=20, value=1)

if st.button("Book plass"):
    if medarbeider:
        # Finn de første ledige plassene
        ledige_slots = [slot for slot in st.session_state.tokens if slot["bruker"] == "Ledig"]

        if len(ledige_slots) >= antall_tokens:
            for i in range(antall_tokens):
                ledige_slots[i]["bruker"] = medarbeider
                ledige_slots[i]["status"] = "Booket"

            st.session_state.logg.insert(0, f"⏱️ {medarbeider} booket {antall_tokens} tokens.")
            st.success(f"Du har booket {antall_tokens} timer!")
            st.rerun()
        else:
            st.error("Det er ikke nok ledige tokens igjen i dag!")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 4. Brukerfunksjoner: Justere/Frigjøre tid (Forskyvning)
st.header("🔧 Endre pågående print")
st.write("Feilet printen, eller ble du ferdig før tiden? Frigjør tokens her så køen rykker frem.")

brikke_id = st.number_input("Hvilket Token-nummer vil du frigjøre/avbryte?", min_value=1, max_value=20, value=1)
kommentar = st.text_input("Obligatorisk årsak/kommentar:")

if st.button("Frigjør token og skyv køen"):
    if kommentar:
        # Finn tokenet som skal frigjøres
        target_token = st.session_state.tokens[brikke_id - 1]
        gammel_bruker = target_token["bruker"]

        if gammel_bruker != "Ledig":
            target_token["bruker"] = "Ledig"
            target_token["status"] = "Ledig"

            # Logg hendelsen
            st.session_state.logg.insert(0, f"⚠️ Token #{brikke_id} frigjort av {gammel_bruker}. Årsak: {kommentar}")
            st.success(f"Token #{brikke_id} er nå ledig for neste mann!")
            st.rerun()
        else:
            st.sidebar.error("Dette tokenet er allerede ledig.")
    else:
        st.warning("Du MÅ skrive en kommentar for å få lov til å endre på tiden!")

# 5. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell")

# Gjør om til en fin tabell-visning
for slot in st.session_state.tokens:
    status_farge = "🟢" if slot["bruker"] == "Ledig" else "🔴"
    st.text(f"{status_farge} Token #{slot['id']:02d} | Kl. {slot['tid']} -> {slot['bruker']}")

# 6. Hendelseslogg (Vises nederst)
st.heading = st.header("📋 Logg (Siste hendelser)")
for hendelse in st.session_state.logg[:5]:  # Viser de 5 siste hendelsene
    st.caption(hendelse)




In [ ]:
Kø V2
import streamlit as st
from datetime import datetime, timedelta
import random

# Tvinger Streamlit til å oppdatere siden automatisk (f.eks. hvert 10. sekund)
# Dette gjør at klokken tikker og gamle timer frigjøres av seg selv.
try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning(
        "Tips: Kjør 'pip install streamlit-autorefresh' i terminalen for at klokken skal oppdatere seg automatisk!")

# 1. Konfigurasjon av siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

# Hent nåtid
naa_tid = datetime.now()
naa_tid_streng = naa_tid.strftime("%H:%M")

# Topplinje med tittel og sanntidsklokke
st.title("🖨️ Bambulab Køen")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

# Husk-melding (Punkt 4)
st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    # Lager 24 tokens for hele døgnet (Punkt 3)
    for i in range(24):
        timer = f"{i:02d}:00"
        er_nattskift = i < 5 or i >= 21  # Kl. 21:00 til 05:00 er nattskift (f.eks. 8 timer totalt)
        st.session_state.tokens.append({
            "id": i + 1,
            "tid": timer,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]  # Noen start-ratings for å vise systemet

# Automatisk frigjøring av gamle timer (Punkt 2)
naa_time = naa_tid.hour
for slot in st.session_state.tokens:
    slot_time = int(slot["tid"].split(":")[0])
    # Hvis timen har passert og den fortsatt er merket som "Booket", frigjør den automatisk
    if slot_time < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 3. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

if st.button("Book plass"):
    if medarbeider:
        # Finn de neste ledige plassene fra og med nå-timen
        ledige_slots = [slot for slot in st.session_state.tokens if
                        slot["bruker"] == "Ledig" and int(slot["tid"].split(":")[0]) >= naa_time]

        if len(ledige_slots) >= antall_tokens:
            valgte_slots = ledige_slots[:antall_tokens]

            # Sjekk om noen av de valgte timene er nattskift (Punkt 3)
            treffer_nattskift = any(slot["nattskift"] for slot in valgte_slots)

            if treffer_nattskift:
                st.warning(
                    "⚠️ **Advarsel:** Printen din går over i nattskiftets vindu (21:00 - 05:00). Har du avtalt med nattskift?")

            # Gjennomfør bookingen
            for slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"

            st.session_state.logg.insert(0,
                                         f"⏱️ {medarbeider} booket {antall_tokens} tokens fra kl. {valgte_slots[0]['tid']}.")
            st.success(f"Suksess! Du har booket {antall_tokens} timer.")
            st.rerun()
        else:
            st.error("Det er ikke nok ledige timer igjen i dag!")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 4. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")
brikke_id = st.number_input("Hvilket Token-nummer vil du avbryte/skyve?", min_value=1, max_value=24, value=1)
kommentar = st.text_input("Obligatorisk årsak/kommentar til forskyvning:")

if st.button("Frigjør token manuelt"):
    if kommentar:
        target_token = st.session_state.tokens[brikke_id - 1]
        gammel_bruker = target_token["bruker"]

        if gammel_bruker != "Ledig":
            target_token["bruker"] = "Ledig"
            target_token["status"] = "Ledig"
            st.session_state.logg.insert(0, f"⚠️ Token #{brikke_id} frigjort av {gammel_bruker}. Årsak: {kommentar}")
            st.success(f"Token #{brikke_id} er frigjort!")
            st.rerun()
        else:
            st.error("Dette tokenet er allerede ledig.")
    else:
        st.warning("Du MÅ skrive en kommentar for å endre på køen!")

# 5. Visning av rutetabellen / Timelisten (Punkt 3)
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    # Definer farge/ikon basert på status og skift
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        tilleggs_tekst = f"-> Booket av {slot['bruker']}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 6. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 7. Feedback-seksjon helt nederst (Punkt 5)
st.header("⭐ Gi tilbakemelding på systemet")

# Regn ut stående rating
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

# Feedback-skjema
stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

In [ ]:
køsystem v3
import streamlit as st
from datetime import datetime, timedelta

# Tvinger Streamlit til å oppdatere siden automatisk hvert 10. sekund
try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk å kjøre 'pip install streamlit-autorefresh' i terminalen!")

# 1. Sette opp siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

naa_tid = datetime.now()
naa_tid_streng = naa_tid.strftime("%H:%M")

st.title("🖨️ Bambulab Tillitsbasert Køsystem")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

# Avdelingsinfo og Husk-meldinger (Punkt 2 og 4)
st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg til **Automasjons Avd.**")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    # Genererer 24 tokens strukturert etter ønske (Punkt 4)
    # Token 1 er kl 01:00, Token 24 er kl 24:00 (00:00)
    for i in range(1, 25):
        time_tall = i
        timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"

        # Nattskift-definisjon: Kl. 21:00 til 07:00 (Token 21, 22, 23, 24, 1, 2, 3, 4, 5, 6, 7)
        er_nattskift = time_tall >= 21 or time_tall <= 7

        st.session_state.tokens.append({
            "id": i,
            "tid": timer_streng,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift,
            "time_verdi": time_tall
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]

# Automatisk frigjøring av gamle timer basert på reell tid
naa_time = naa_tid.hour
# Hvis klokken er 00:XX, regner vi det som time 24 i vårt system
if naa_time == 0:
    naa_time = 24

for slot in st.session_state.tokens:
    # Hvis timen har passert og den er booket, frigjør den
    if slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 3. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

# Hakeboks for nattskift-godkjenning (Punkt 1)
godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")

if st.button("Book plass"):
    if medarbeider:
        # Filtrer tilgjengelige slots som er i fremtiden (eller gjeldende time)
        fremtidige_slots = [slot for slot in st.session_state.tokens if
                            slot["bruker"] == "Ledig" and slot["time_verdi"] >= naa_time]

        # Hvis de IKKE har huket av for nattskift, fjerner vi nattskifttimene fra listen deres (Punkt 1)
        if not godkjent_nattskift:
            tilgjengelige_slots = [slot for slot in fremtidige_slots if not slot["nattskift"]]
        else:
            tilgjengelige_slots = fremtidige_slots

        if len(tilgjengelige_slots) >= antall_tokens:
            valgte_slots = tilgjengelige_slots[:antall_tokens]

            # Gjennomfør bookingen
            for slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"

            st.session_state.logg.insert(0,
                                         f"⏱️ {medarbeider} booket {antall_tokens} tokens fra kl. {valgte_slots[0]['tid']}.")
            st.success(f"Suksess! Du har booket {antall_tokens} timer fra klokken {valgte_slots[0]['tid']}.")
            st.rerun()
        else:
            if not godkjent_nattskift:
                st.error(
                    "Ikke nok ledige dagtimer! Hvis du må printe inn i nattskiftet, må du huke av for '🌙 Jeg har avtalt med nattskift'.")
            else:
                st.error("Det er ikke nok ledige timer igjen i dag!")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 4. Brukerfunksjoner: Justere/Frigjøre tid manuelt (Punkt 3)
st.header("🔧 Endre pågående print")

# Lager en liste med tekst for dropdown-menyen, f.eks: "Token #8 (Kl. 08:00) - Booket av Ole"
avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        avbestillings_valg.append(f"Token #{slot['id']} (Kl. {slot['tid']}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        # Henter ut Token ID fra teksten (f.eks tallet etter #)
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort og klar for neste mann!")
        st.rerun()
    else:
        st.warning("Du må BÅDE velge et booket token fra menyen og skrive en kommentar!")

# 5. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        tilleggs_tekst = f"-> Booket av {slot['bruker']}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 6. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 7. Feedback-seksjon helt nederst
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

st.write("---")

# 8. Signatur (Punkt 5)
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)

In [ ]:
køsystem v4
import streamlit as st
from datetime import datetime

# Tvinger Streamlit til å oppdatere siden automatisk hvert 10. sekund
try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk å kjøre 'pip install streamlit-autorefresh' i terminalen!")

# 1. Sette opp siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

naa_tid = datetime.now()
naa_tid_streng = naa_tid.strftime("%H:%M")

st.title("🖨️ Bambulab Tillitsbasert Køsystem")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

# Avdelingsinfo og Husk-meldinger
st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg til **Automasjons Avd.**")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    # Genererer 24 tokens strukturert etter ønske
    for i in range(1, 25):
        time_tall = i
        timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"

        # KUN Token 1 til 7 er nattskift (Punkt 1)
        er_nattskift = 1 <= i <= 7

        st.session_state.tokens.append({
            "id": i,
            "tid": timer_streng,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift,
            "time_verdi": time_tall
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]

# Automatisk frigjøring av gamle timer basert på reell tid
naa_time = naa_tid.hour
if naa_time == 0:
    naa_time = 24

for slot in st.session_state.tokens:
    if slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 3. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

# Hakeboks for nattskift-godkjenning
godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")

if st.button("Book plass"):
    if medarbeider:
        # Lag en liste over tokens sortert kronologisk fra NÅ og fremover (inkludert neste dag)
        # Dette gjør at hvis klokken er 23, sjekkes time 23, 24, 1, 2, 3... osv.
        kronologisk_koe = []
        for i in range(24):
            sjekk_time = ((naa_time - 1 + i) % 24) + 1
            # Finn matchende token i session_state
            token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
            kronologisk_koe.append(token_obj)

        # Filtrer køen basert på om de har lov til å bruke nattskifttimer eller ikke (Punkt 2)
        filtrert_koe = []
        for slot in kronologisk_koe:
            if slot["status"] == "Ledig":
                if slot["nattskift"] and not godkjent_nattskift:
                    continue  # Hopper over nattskift-tokens hvis boks ikke er krysset av
                filtrert_koe.append(slot)

        # Sjekk om vi fant nok ledige slots etter filtreringen
        if len(filtrert_koe) >= antall_tokens:
            valgte_slots = filtrert_koe[:antall_tokens]

            # Gjennomfør bookingen
            for slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"

            st.session_state.logg.insert(0,
                                         f"⏱️ {medarbeider} booket {antall_tokens} tokens. Første ledige ble: Token #{valgte_slots[0]['id']} (Kl. {valgte_slots[0]['tid']}).")
            st.success(
                f"Suksess! Du har booket {antall_tokens} timer. Første ledige time ble klokken {valgte_slots[0]['tid']}.")
            st.rerun()
        else:
            st.error("Det er ikke nok ledige timer tilgjengelig som matcher dine valg!")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 4. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")

avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        avbestillings_valg.append(f"Token #{slot['id']} (Kl. {slot['tid']}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort!")
        st.rerun()
    else:
        st.warning("Du må BÅDE velge et booket token fra menyen og skrive en kommentar!")

# 5. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        tilleggs_tekst = f"-> Booket av {slot['bruker']}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 6. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 7. Feedback-seksjon helt nederst
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

st.write("---")

# 8. Signatur
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)

In [ ]:
kø v5 nesten perfekt
import streamlit as st
from datetime import datetime

# Tvinger Streamlit til å oppdatere siden automatisk hvert 10. sekund
try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk å kjøre 'pip install streamlit-autorefresh' i terminalen!")

# 1. Sette opp siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

naa_tid = datetime.now()
naa_tid_streng = naa_tid.strftime("%H:%M")

st.title("🖨️ Bambulab Tillitsbasert Køsystem")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

# Avdelingsinfo og Husk-meldinger
st.info("💡 **HUSK:** Legg fra deg de feceske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg til **Automasjons Avd.**")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    # Genererer 24 tokens strukturert etter ønske
    for i in range(1, 25):
        time_tall = i
        timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"

        # KUN Token 1 til 7 er nattskift
        er_nattskift = 1 <= i <= 7

        st.session_state.tokens.append({
            "id": i,
            "tid": timer_streng,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift,
            "time_verdi": time_tall,
            "dag": "i_dag"  # Standard er at de tilhører i dag frem til de blir booket for i morgen
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]

# 3. Automatisk frigjøring av gamle timer (KUN for brikker merket med "i_dag")
naa_time = naa_tid.hour
if naa_time == 0:
    naa_time = 24

for slot in st.session_state.tokens:
    # KRITISK FIKS: Den sletter NÅ kun hvis slotten faktisk tilhører "i_dag"
    if slot["dag"] == "i_dag" and slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 4. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

col1, col2 = st.columns(2)
with col1:
    godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")
with col2:
    book_for_i_morgen = st.checkbox("📅 Book for i morgen (fra Token 8 / 08:00)")

if st.button("Sjekk tilgjengelighet og book plass"):
    if medarbeider:
        kronologisk_soke_koe = []

        if book_for_i_morgen:
            # SØK FOR I MORGEN: Starter på Token 8 og går 24 timer fremover
            for i in range(24):
                sjekk_time = ((8 - 1 + i) % 24) + 1
                token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
                kronologisk_soke_koe.append(token_obj)
        else:
            # SØK FOR I DAG: Starter fra nå-tid
            for i in range(24):
                sjekk_time = ((naa_time - 1 + i) % 24) + 1
                token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
                kronologisk_soke_koe.append(token_obj)

        # Filtrer basert på ledig status og nattskift-knapp
        filtrert_koe = []
        for slot in kronologisk_soke_koe:
            if slot["status"] == "Ledig":
                if slot["nattskift"] and not godkjent_nattskift:
                    continue
                filtrert_koe.append(slot)

        # Sjekk om vi har nok plasser
        if len(filtrert_koe) >= antall_tokens:
            valgte_slots = filtrert_koe[:antall_tokens]

            # Utfør bookingen og merk med riktig dag!
            for slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"
                slot["dag"] = "i_morgen" if book_for_i_morgen else "i_dag"  # FIKS: Beskytter mot auto-sletting

            dag_tekst = "i morgen" if book_for_i_morgen else "i dag"
            st.session_state.logg.insert(0,
                                         f"⏱️ {medarbeider} booket {antall_tokens} tokens for {dag_tekst}. Start: Token #{valgte_slots[0]['id']} ({valgte_slots[0]['tid']}).")
            st.success(
                f"Suksess! Du har booket {antall_tokens} timer for {dag_tekst}. Første ledige ble Token #{valgte_slots[0]['id']} kl {valgte_slots[0]['tid']}.")
            st.rerun()
        else:
            st.error("Det er ikke nok ledige timer tilgjengelig for valgte tidsperiode.")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 5. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")

avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        dag_label = "I morgen" if slot["dag"] == "i_morgen" else "I dag"
        avbestillings_valg.append(f"Token #{slot['id']} ({slot['tid']} - {dag_label}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        target_token["dag"] = "i_dag"  # Nullstill til i dag
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort!")
        st.rerun()
    else:
        st.warning("Du må både velge et token og skrive en kommentar!")

# 6. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        dag_info = " [I MORGEN]" if slot["dag"] == "i_morgen" else ""
        tilleggs_tekst = f"-> Booket av {slot['bruker']}{dag_info}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 7. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 8. Feedback-seksjon
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

st.write("---")

# 9. Signatur
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)

In [ ]:
kø vibe 6?
import streamlit as st
from datetime import datetime

# Tvinger Streamlit til å oppdatere siden automatisk hvert 10. sekund
try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk å kjøre 'pip install streamlit-autorefresh' i terminalen!")

# 1. Sette opp siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

naa_tid = datetime.now()
naa_tid_streng = naa_tid.strftime("%H:%M")

st.title("🖨️ Bambulab Tillitsbasert Køsystem")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

# Avdelingsinfo og Husk-meldinger
st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg til **Automasjons Avd.**")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    # Genererer 24 tokens strukturert etter ønske
    for i in range(1, 25):
        time_tall = i
        timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"

        # KUN Token 1 til 7 er nattskift
        er_nattskift = 1 <= i <= 7

        st.session_state.tokens.append({
            "id": i,
            "tid": timer_streng,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift,
            "time_verdi": time_tall,
            "dag": "i_dag"
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]

# 3. Automatisk frigjøring av gamle timer (KUN for brikker merket med "i_dag")
naa_time = naa_tid.hour
if naa_time == 0:
    naa_time = 24

for slot in st.session_state.tokens:
    if slot["dag"] == "i_dag" and slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 4. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

col1, col2 = st.columns(2)
with col1:
    godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")
with col2:
    book_for_i_morgen = st.checkbox("📅 Book for i morgen (fra Token 8 / 08:00)")

if st.button("Sjekk tilgjengelighet og book plass"):
    if medarbeider:
        kronologisk_soke_koe = []

        # Setter startpunktet for søket basert på om det er for i dag eller i morgen
        if book_for_i_morgen:
            start_soke_time = 8  # Starter alltid på Token 8 (08:00) neste dag
        else:
            start_soke_time = naa_time  # Starter fra nå-timen i dag

        # Lag en søkekø som ruller over 24 sammenhengende timer
        for i in range(24):
            sjekk_time = ((start_soke_time - 1 + i) % 24) + 1
            token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
            kronologisk_soke_koe.append(token_obj)

        # Filtrer søkekøen basert på status og om de har lov til å bruke nattskift
        filtrert_koe = []
        for index, slot in enumerate(kronologisk_soke_koe):
            if slot["status"] == "Ledig":
                # KRITISK ENDRING: Nattskift-sjekken gjelder nå uansett om det rulles over eller bookes direkte.
                # Hvis brikken er et nattskift-token (1-7) og knappen IKKE er huket av, blir den hoppet over.
                if slot["nattskift"] and not godkjent_nattskift:
                    continue
                filtrert_koe.append((index, slot))

        # Sjekk om vi finner nok tilgjengelige tokens totalt
        if len(filtrert_koe) >= antall_tokens:
            valgte_par = filtrert_koe[:antall_tokens]

            # Sjekk om tidsrekken brytes (vi ønsker i utgangspunktet sammenhengende tid hvis mulig)
            # Hvis det blir et "hull" pga et nattskift som hoppes over, vil den bare ta de neste ledige dagtimene.

            # Utfør bookingen
            for soke_index, slot in valgte_par:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"

                # Hvis vi søker for i morgen, er det uansett i morgen.
                # Hvis vi søker for i dag, men soke_index + start_soke_time bikker 24, betyr det at timen har rullet over midnatt!
                if book_for_i_morgen:
                    slot["dag"] = "i_morgen"
                else:
                    # Sjekker om brikken ligger etter midnatt i denne spesifikke søke-runden
                    if start_soke_time + soke_index > 24:
                        slot["dag"] = "i_morgen"
                    else:
                        slot["dag"] = "i_dag"

            første_token = valgte_par[0][1]
            st.session_state.logg.insert(0,
                                         f"⏱️ {medarbeider} booket {antall_tokens} tokens. Start: Token #{første_token['id']} ({første_token['tid']}).")
            st.success(
                f"Suksess! Du har booket {antall_tokens} timer. Starttid: Token #{første_token['id']} klokken {første_token['tid']}.")
            st.rerun()
        else:
            st.error("Det er ikke nok ledige timer tilgjengelig som matcher dine valg. Sjekk timetabellen under.")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 5. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")

avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        dag_label = "I morgen" if slot["dag"] == "i_morgen" else "I dag"
        avbestillings_valg.append(f"Token #{slot['id']} ({slot['tid']} - {dag_label}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        target_token["dag"] = "i_dag"  # Nullstill
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort!")
        st.rerun()
    else:
        st.warning("Du må både velge et token og skrive en kommentar!")

# 6. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        dag_info = " [I MORGEN]" if slot["dag"] == "i_morgen" else ""
        tilleggs_tekst = f"-> Booket av {slot['bruker']}{dag_info}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 7. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 8. Feedback-seksjon
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

st.write("---")

# 9. Signatur
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)


In [ ]:
e vi virkelig på 7 versjonen?
import streamlit as st
from datetime import datetime

# Tvinger Streamlit til å oppdatere siden automatisk hvert 10. sekund
try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk å kjøre 'pip install streamlit-autorefresh' i terminalen!")

# 1. Sette opp siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

naa_tid = datetime.now()
naa_tid_streng = naa_tid.strftime("%H:%M")

st.title("🖨️ Bambulab køen for GK MEK verksted")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

# Avdelingsinfo og Husk-meldinger
st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg to **Automasjons Avd.**")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    # Genererer 24 tokens strukturert etter ønske
    for i in range(1, 25):
        time_tall = i
        timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"

        # KUN Token 1 til 7 er nattskift
        er_nattskift = 1 <= i <= 7

        st.session_state.tokens.append({
            "id": i,
            "tid": timer_streng,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift,
            "time_verdi": time_tall,
            "dag": "i_dag"
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]

# 3. Automatisk frigjøring av gamle timer (KUN for brikker merket med "i_dag")
naa_time = naa_tid.hour
if naa_time == 0:
    naa_time = 24

for slot in st.session_state.tokens:
    if slot["dag"] == "i_dag" and slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 4. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

col1, col2 = st.columns(2)
with col1:
    godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")
with col2:
    book_for_i_morgen = st.checkbox("📅 Book for i morgen (fra Token 8 / 08:00)")

if st.button("Sjekk tilgjengelighet og book plass"):
    if medarbeider:
        kronologisk_soke_koe = []

        if book_for_i_morgen:
            start_soke_time = 8  # Starter på Token 8 neste dag
        else:
            start_soke_time = naa_time  # Starter fra nå-timen i dag

        # Lag en uavbrutt tidsrekke på 24 sammenhengende timer fremover
        for i in range(24):
            sjekk_time = ((start_soke_time - 1 + i) % 24) + 1
            token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
            kronologisk_soke_koe.append((i, token_obj))

        # Sjekk og bygg listen over tillatte timer på rad
        valgte_slots = []
        antall_frigjort_pga_nattskift = 0

        for soke_index, slot in kronologisk_soke_koe:
            # Stopp letingen helt hvis vi har funnet nok timer
            if len(valgte_slots) == antall_tokens:
                break

            if slot["status"] == "Ledig":
                # KRITISK ENDRING: Hvis brikken er nattskift, og de IKKE har huket av:
                # Da KUTTES bookingen her. De resterende timene de ba om blir "frigjort/avvist".
                if slot["nattskift"] and not godkjent_nattskift:
                    antall_frigjort_pga_nattskift = antall_tokens - len(valgte_slots)
                    break  # Går rett ut av løkken, ingen flere timer tillates!

                valgte_slots.append((soke_index, slot))
            else:
                # Hvis en time midt i rekken allerede er booket av noen andre, må vi også stoppe (sammenhengende tid)
                break

        # Gjennomfør booking for de timene som faktisk ble godkjent
        if len(valgte_slots) > 0:
            for soke_index, slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"

                if book_for_i_morgen:
                    slot["dag"] = "i_morgen"
                else:
                    if start_soke_time + soke_index > 24:
                        slot["dag"] = "i_morgen"
                    else:
                        slot["dag"] = "i_dag"

            første_token = valgte_slots[0][1]
            siste_token = valgte_slots[-1][1]

            # Loggføring og tilbakemelding
            logg_melding = f"⏱️ {medarbeider} booket {len(valgte_slots)} tokens (Kl. {første_token['tid']} til {siste_token['tid']})."
            if antall_frigjort_pga_nattskift > 0:
                logg_melding += f" ⚠️ {antall_frigjort_pga_nattskift} timer automatisk frigjort pga manglende nattskift-avtale."
                st.warning(
                    f"Du fikk booket {len(valgte_slots)} timer frem til nattskiftet startet, men de siste {antall_frigjort_pga_nattskift} timene ble automatisk frigjort/avvist fordi du ikke har huket av for avtale med nattskift!")
            else:
                st.success(f"Suksess! Du har booket {len(valgte_slots)} sammenhengende timer.")

            st.session_state.logg.insert(0, logg_melding)
            st.rerun()
        else:
            st.error(
                "Kunne ikke booke noen plasser. Enten er neste time opptatt, eller så starter nattskiftet umiddelbart og blokkerer søket ditt.")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 5. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")

avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        dag_label = "I morgen" if slot["dag"] == "i_morgen" else "I dag"
        avbestillings_valg.append(f"Token #{slot['id']} ({slot['tid']} - {dag_label}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        target_token["dag"] = "i_dag"
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort!")
        st.rerun()
    else:
        st.warning("Du må både velge et token og skrive en kommentar!")

# 6. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        dag_info = " [I MORGEN]" if slot["dag"] == "i_morgen" else ""
        tilleggs_tekst = f"-> Booket av {slot['bruker']}{dag_info}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 7. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 8. Feedback-seksjon
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

st.write("---")

# 9. Signatur
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)

In [ ]:
fungerer bra, men er gullfisk!
import streamlit as st
from datetime import datetime
import pytz

try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk å kjøre 'pip install streamlit-autorefresh' i terminalen!")

# 1. Sette opp siden
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

norsk_tidssone = pytz.timezone("Europe/Oslo")
naa_tid = datetime.now(norsk_tidssone)
naa_tid_streng = naa_tid.strftime("%H:%M")

st.title("🖨️ Bambulab Tillitsbasert Køsystem")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg til **Automasjons Avd.**")

# 2. Sette opp "State" (Dataminne)
if "tokens" not in st.session_state:
    st.session_state.tokens = []
    for i in range(1, 25):
        time_tall = i
        timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"
        er_nattskift = 1 <= i <= 7

        st.session_state.tokens.append({
            "id": i,
            "tid": timer_streng,
            "bruker": "Ledig",
            "status": "Ledig",
            "nattskift": er_nattskift,
            "time_verdi": time_tall,
            "dag": "i_dag"
        })

if "logg" not in st.session_state:
    st.session_state.logg = ["Systemet startet. Alt klart for print!"]

if "ratings" not in st.session_state:
    st.session_state.ratings = [5, 5, 4]

# 3. Automatisk frigjøring av gamle timer
naa_time = naa_tid.hour
if naa_time == 0:
    naa_time = 24

for slot in st.session_state.tokens:
    if slot["dag"] == "i_dag" and slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")

# 4. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

col1, col2 = st.columns(2)
with col1:
    godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")
with col2:
    book_for_i_morgen = st.checkbox("📅 Book for i morgen (fra Token 8 / 08:00)")

if st.button("Sjekk tilgjengelighet og book plass"):
    if medarbeider:
        kronologisk_soke_koe = []

        if book_for_i_morgen:
            start_soke_time = 8
        else:
            start_soke_time = naa_time

        # Lag en uavbrutt tidsrekke på 24 sammenhengende timer fremover
        for i in range(24):
            sjekk_time = ((start_soke_time - 1 + i) % 24) + 1
            token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
            kronologisk_soke_koe.append((i, token_obj))

        # NY LOGIKK: Finn ALLE tilgjengelige og lovlige timer i rekkefølgen
        lovlige_og_ledige = []
        antall_avvist_pga_nattskift = 0

        for soke_index, slot in kronologisk_soke_koe:
            if slot["status"] == "Ledig":
                if slot["nattskift"] and not godkjent_nattskift:
                    # Hvis vi treffer nattskiftet uten avtale, lagrer vi hvor mange timer vi manglet og stopper
                    if len(lovlige_og_ledige) > 0:
                        antall_avvist_pga_nattskift = antall_tokens - len(lovlige_og_ledige)
                    break
                lovlige_og_ledige.append((soke_index, slot))
            else:
                # Hvis vi allerede har begynt å samle opp en sammenhengende blokk,
                # men treffer en opptatt time, må vi stoppe for å holde tiden sammenhengende.
                if len(lovlige_og_ledige) > 0:
                    break
                # Hvis vi IKKE har funnet noen ledige timer ennå, fortsetter vi bare å lete fremover i køen!
                continue

        # Sjekk om vi fant noen brukbare timer i det hele tatt
        if len(lovlige_og_ledige) > 0:
            valgte_slots = lovlige_og_ledige[:antall_tokens]

            # Utfør bookingen
            for soke_index, slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"

                if book_for_i_morgen:
                    slot["dag"] = "i_morgen"
                else:
                    if start_soke_time + soke_index > 24:
                        slot["dag"] = "i_morgen"
                    else:
                        slot["dag"] = "i_dag"

            første_token = valgte_slots[0][1]
            siste_token = valgte_slots[-1][1]

            logg_melding = f"⏱️ {medarbeider} booket {len(valgte_slots)} tokens (Kl. {første_token['tid']} til {siste_token['tid']})."

            if antall_avvist_pga_nattskift > 0:
                logg_melding += f" ⚠️ {antall_avvist_pga_nattskift} timer automatisk frigjort pga nattskift-sperre."
                st.warning(
                    f"Du fikk booket {len(valgte_slots)} timer, men de siste {antall_avvist_pga_nattskift} timene ble avvist fordi du krasjet med nattskiftet uten avtale!")
            else:
                st.success(
                    f"Suksess! Du har booket {len(valgte_slots)} sammenhengende timer fra klokken {første_token['tid']}.")

            st.session_state.logg.insert(0, logg_melding)
            st.rerun()
        else:
            st.error(
                "Kunne ikke finne noen ledige timer etter hverandre. Sjekk timetabellen under for å finne et ledig vindu.")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 5. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")

avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        dag_label = "I morgen" if slot["dag"] == "i_morgen" else "I dag"
        avbestillings_valg.append(f"Token #{slot['id']} ({slot['tid']} - {dag_label}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        target_token["dag"] = "i_dag"
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort!")
        st.rerun()
    else:
        st.warning("Du må både velge et token og skrive en kommentar!")

# 6. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        dag_info = " [I MORGEN]" if slot["dag"] == "i_morgen" else ""
        tilleggs_tekst = f"-> Booket av {slot['bruker']}{dag_info}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 7. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 8. Feedback-seksjon
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    st.rerun()

st.write("---")

# 9. Signatur
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)

In [ ]:
eg vet ikke lengre vilke versjon vi e på! men no nærmer det seg for alvor! funker perfekt
    - ingen lagring av rating og komentarer
import streamlit as st
from datetime import datetime
import pytz
import json
import os

try:
    from streamlit_autorefresh import st_autorefresh

    st_autorefresh(interval=10000, key="daterefresh")
except ImportError:
    st.warning("Husk 'pip install streamlit-autorefresh'")

# 1. Sette opp siden og tid
st.set_page_config(page_title="Bambulab Køsystem", page_icon="🖨️", layout="centered")

norsk_tidssone = pytz.timezone("Europe/Oslo")
naa_tid = datetime.now(norsk_tidssone)
naa_tid_streng = naa_tid.strftime("%H:%M")
dagens_dato_streng = naa_tid.strftime("%Y-%m-%d")

st.title("🖨️ Bambulab print kø MEK ")
st.subheader(f"🕒 Gjeldende klokkeslett: {naa_tid_streng}")

st.info("💡 **HUSK:** Legg fra deg de fysiske tokens ved printeren med en gang printen din starter!")
st.error("🔧 **Printerkrøll eller misnøye?** Hvis du ikke klarer å fikse det selv, henvend deg til **Automasjons Avd.**")

# --- NYTT: LOGIKK FOR PERMANENT FIL-MINNE ---
FILNAVN = "koe_data.json"


def last_lagrede_data():
    """Henter data fra JSON-filen hvis den eksisterer og er fra i dag."""
    if os.path.exists(FILNAVN):
        try:
            with open(FILNAVN, "r", encoding="utf-8") as f:
                data = json.load(f)
                # Sjekk om filen er fra i dag (24-timers automatisk nullstilling)
                if data.get("dato") == dagens_dato_streng:
                    return data
        except Exception:
            pass
    return None


def lagre_data_til_fil():
    """Lagrer gjeldende status til JSON-filen."""
    data_som_skal_lagres = {
        "dato": dagens_dato_streng,
        "tokens": st.session_state.tokens,
        "logg": st.session_state.logg,
        "ratings": st.session_state.ratings
    }
    with open(FILNAVN, "w", encoding="utf-8") as f:
        json.dump(data_som_skal_lagres, f, ensure_ascii=False, indent=4)


# Last inn minnet hvis det finnes
lagret_minne = last_lagrede_data()

# 2. Sette opp "State" basert på fil-minnet
if "tokens" not in st.session_state:
    if lagret_minne:
        st.session_state.tokens = lagret_minne["tokens"]
        st.session_state.logg = lagret_minne["logg"]
        st.session_state.ratings = lagret_minne["ratings"]
    else:
        # Hvis ingen fil finnes (eller den er utdatert), lag ny blank dag
        st.session_state.tokens = []
        for i in range(1, 25):
            time_tall = i
            timer_streng = f"{time_tall:02d}:00" if time_tall < 24 else "24:00"
            er_nattskift = 1 <= i <= 7
            st.session_state.tokens.append({
                "id": i,
                "tid": timer_streng,
                "bruker": "Ledig",
                "status": "Ledig",
                "nattskift": er_nattskift,
                "time_verdi": time_tall,
                "dag": "i_dag"
            })
        st.session_state.logg = ["Systemet startet. Alt klart for print!"]
        st.session_state.ratings = [5, 5, 4]
        lagre_data_til_fil()

# 3. Automatisk frigjøring av gamle timer
naa_time = naa_tid.hour
if naa_time == 0:
    naa_time = 24

endring_skjedd = False
for slot in st.session_state.tokens:
    if slot["dag"] == "i_dag" and slot["time_verdi"] < naa_time and slot["status"] == "Booket":
        gammel_bruker = slot["bruker"]
        slot["bruker"] = "Ledig"
        slot["status"] = "Ledig"
        st.session_state.logg.insert(0,
                                     f"🤖 Automatisk frigjort: Tiden for Token #{slot['id']} ({slot['tid']}) har passert.")
        endring_skjedd = True

if endring_skjedd:
    lagre_data_til_fil()

# 4. Brukerfunksjoner: Booke tokens
st.header("🛒 Book printertid")
medarbeider = st.text_input("Ditt navn:", placeholder="f.eks. Mathias")
antall_tokens = st.number_input("Hvor mange timer/tokens trenger du?", min_value=1, max_value=24, value=1)

col1, col2 = st.columns(2)
with col1:
    godkjent_nattskift = st.checkbox("🌙 Jeg har avtalt med nattskift")
with col2:
    book_for_i_morgen = st.checkbox("📅 Book for i morgen (fra Token 8 / 08:00)")

if st.button("Sjekk tilgjengelighet og book plass"):
    if medarbeider:
        kronologisk_soke_koe = []
        if book_for_i_morgen:
            start_soke_time = 8
        else:
            start_soke_time = naa_time

        for i in range(24):
            sjekk_time = ((start_soke_time - 1 + i) % 24) + 1
            token_obj = next(slot for slot in st.session_state.tokens if slot["time_verdi"] == sjekk_time)
            kronologisk_soke_koe.append((i, token_obj))

        lovlige_og_ledige = []
        antall_avvist_pga_nattskift = 0

        for soke_index, slot in kronologisk_soke_koe:
            if len(lovlige_og_ledige) == antall_tokens:
                break
            if slot["status"] == "Ledig":
                if slot["nattskift"] and not godkjent_nattskift:
                    if len(lovlige_og_ledige) > 0:
                        antall_avvist_pga_nattskift = antall_tokens - len(lovlige_og_ledige)
                    break
                lovlige_og_ledige.append((soke_index, slot))
            else:
                if len(lovlige_og_ledige) > 0:
                    break
                continue

        if len(lovlige_og_ledige) > 0:
            valgte_slots = lovlige_og_ledige[:antall_tokens]
            for soke_index, slot in valgte_slots:
                slot["bruker"] = medarbeider
                slot["status"] = "Booket"
                if book_for_i_morgen:
                    slot["dag"] = "i_morgen"
                else:
                    if start_soke_time + soke_index > 24:
                        slot["dag"] = "i_morgen"
                    else:
                        slot["dag"] = "i_dag"

            første_token = valgte_slots[0][1]
            siste_token = valgte_slots[-1][1]
            logg_melding = f"⏱️ {medarbeider} booket {len(valgte_slots)} tokens (Kl. {første_token['tid']} til {siste_token['tid']})."

            if antall_avvist_pga_nattskift > 0:
                logg_melding += f" ⚠️ {antall_avvist_pga_nattskift} timer automatisk frigjort pga nattskift-sperre."
                st.warning(
                    f"Du fikk booket {len(valgte_slots)} timer, men de siste {antall_avvist_pga_nattskift} timene ble avvist!")
            else:
                st.success(f"Suksess! Du har booket {len(valgte_slots)} sammenhengende timer.")

            st.session_state.logg.insert(0, logg_melding)
            lagre_data_til_fil()  # SPESIKT MINNE-STEG
            st.rerun()
        else:
            st.error("Kunne ikke finne noen ledige timer etter hverandre.")
    else:
        st.warning("Du må skrive inn navnet ditt først.")

# 5. Brukerfunksjoner: Justere/Frigjøre tid manuelt
st.header("🔧 Endre pågående print")

avbestillings_valg = ["-- Velg et token --"]
for slot in st.session_state.tokens:
    if slot["status"] == "Booket":
        dag_label = "I morgen" if slot["dag"] == "i_morgen" else "I dag"
        avbestillings_valg.append(f"Token #{slot['id']} ({slot['tid']} - {dag_label}) - {slot['bruker']}")

valgt_streng = st.selectbox("Hvilket Token vil du avbryte/skyve?", options=avbestillings_valg)
kommentar = st.text_input("Obligatorisk årsak/kommentar til endringen:")

if st.button("Frigjør token manuelt"):
    if valgt_streng != "-- Velg et token --" and kommentar:
        token_id_valgt = int(valgt_streng.split("#")[1].split(" ")[0])

        target_token = st.session_state.tokens[token_id_valgt - 1]
        gammel_bruker = target_token["bruker"]

        target_token["bruker"] = "Ledig"
        target_token["status"] = "Ledig"
        target_token["dag"] = "i_dag"
        st.session_state.logg.insert(0, f"⚠️ Token #{token_id_valgt} frigjort av {gammel_bruker}. Årsak: {kommentar}")
        st.success(f"Token #{token_id_valgt} er frigjort!")
        lagre_data_til_fil()  # SPESIKT MINNE-STEG
        st.rerun()
    else:
        st.warning("Du må både velge et token og skrive en kommentar!")

# 6. Visning av rutetabellen / Timelisten
st.header("📅 Dagens Timetabell (24 Timer)")

for slot in st.session_state.tokens:
    if slot["bruker"] != "Ledig":
        status_ikon = "🔴"
        dag_info = " [I MORGEN]" if slot["dag"] == "i_morgen" else ""
        tilleggs_tekst = f"-> Booket av {slot['bruker']}{dag_info}"
    elif slot["nattskift"]:
        status_ikon = "🌙"
        tilleggs_tekst = "(Ledig - KUN FOR NATTSKIFT)"
    else:
        status_ikon = "🟢"
        tilleggs_tekst = "(Ledig)"

    st.text(f"{status_ikon} Token #{slot['id']:02d} | Kl. {slot['tid']} {tilleggs_tekst}")

# 7. Hendelseslogg
st.header("📋 Siste hendelser")
for hendelse in st.session_state.logg[:3]:
    st.caption(hendelse)

st.write("---")

# 8. Feedback-seksjon
st.header("⭐ Gi tilbakemelding på systemet")
gjennomsnitt = sum(st.session_state.ratings) / len(st.session_state.ratings)
st.subheader(f"Gjennomsnittlig rating: {gjennomsnitt:.1f} / 5.0 stjerner ({len(st.session_state.ratings)} stemmer)")

stjerner = st.slider("Hvor mange stjerner gir du dette køsystemet?", min_value=1, max_value=5, value=5)
tilbakemelding_tekst = st.text_area("Kommentar (valgfri):", placeholder="Hva kan forbedres?")

if st.button("Send tilbakemelding"):
    st.session_state.ratings.append(stjerner)
    st.success("Tusen takk for tilbakemeldingen din!")
    lagre_data_til_fil()  # SPESIKT MINNE-STEG
    st.rerun()

st.write("---")

# 9. Signatur
st.markdown("<center><h4>Vibet av EY-FLYTTDG 🤘</h4></center>", unsafe_allow_html=True)